# Chapter 3 Lab: Generalization, Leakage, and Evaluation Discipline (`ch02b`)

This lab focuses on generalization: the gap between training and test performance. We'll see overfitting in action, use cross-validation to select models, perform sanity checks, and plot learning curves.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, learning_curve, KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

%matplotlib inline

# Seed for reproducibility
np.random.seed(42)

## Section 1: Overfitting Demo with Polynomial Regression

Let's fit polynomials of increasing complexity to a simple sine curve with noise. As complexity grows, training error shrinks but test error eventually rises: the classic U-shape of overfitting.

In [ ]:
# Generate a small training set: 20 noisy sine points
np.random.seed(42)
n_train = 20
X_train = np.linspace(0, 2*np.pi, n_train).reshape(-1, 1)
y_train = np.sin(X_train.ravel()) + np.random.normal(0, 0.15, n_train)

# Generate a test set: dense, noiseless sine curve
X_test = np.linspace(0, 2*np.pi, 200).reshape(-1, 1)
y_test = np.sin(X_test.ravel())

print(f"Training set: {len(X_train)} noisy points")
print(f"Test set: {len(X_test)} noiseless points (true sine curve)")

In [ ]:
# Fit polynomials of different degrees
degrees = [1, 3, 5, 10, 15]
models = {}
train_errors = []
test_errors = []

for degree in degrees:
    # Create and fit polynomial regression
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('ridge', Ridge(alpha=0.001))  # Tiny regularization
    ])
    model.fit(X_train, y_train)
    models[degree] = model
    
    # Compute errors
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    
    train_errors.append(np.sqrt(train_mse))  # RMSE
    test_errors.append(np.sqrt(test_mse))
    
    print(f"Degree {degree:2d}: Train RMSE = {np.sqrt(train_mse):.4f}, Test RMSE = {np.sqrt(test_mse):.4f}")

print(f"\nNotice: Training error keeps shrinking, but test error increases after degree 5.")
print(f"This is overfitting—the model memorizes noise instead of learning the true sine pattern.")

## Section 2: Visualizing the Overfitting U-Curve

In [ ]:
# Plot train vs test error vs model complexity
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Error vs complexity
ax1.plot(degrees, train_errors, marker='o', linewidth=2.5, markersize=8,
         color='#2E86AB', label='Training Error', alpha=0.8)
ax1.plot(degrees, test_errors, marker='s', linewidth=2.5, markersize=8,
         color='#A23B72', label='Test Error', alpha=0.8)

# Shade overfitting region
ax1.axvspan(5, 15, alpha=0.1, color='red', label='Overfitting Region')

ax1.set_xlabel('Polynomial Degree', fontsize=12, fontweight='bold')
ax1.set_ylabel('Root Mean Squared Error (RMSE)', fontsize=12, fontweight='bold')
ax1.set_title('The U-Curve: Overfitting as Complexity Grows', fontsize=13, fontweight='bold')
ax1.set_xticks(degrees)
ax1.legend(fontsize=11, loc='upper left')
ax1.grid(alpha=0.3, linestyle='--')

# Right plot: Fitted polynomials
X_plot = np.linspace(0, 2*np.pi, 300).reshape(-1, 1)
colors = ['#2E86AB', '#A23B72', '#F77F00', '#E63946', '#C1121F']

for i, degree in enumerate(degrees):
    y_plot = models[degree].predict(X_plot)
    ax2.plot(X_plot, y_plot, linewidth=2, label=f'Degree {degree}', color=colors[i], alpha=0.7)

# Plot true function and training data
y_true = np.sin(X_plot.ravel())
ax2.plot(X_plot, y_true, 'k--', linewidth=2.5, label='True sin(x)', alpha=0.7)
ax2.scatter(X_train, y_train, color='black', s=40, zorder=5, label='Training data (noisy)', alpha=0.6)

ax2.set_xlabel('x', fontsize=12, fontweight='bold')
ax2.set_ylabel('y', fontsize=12, fontweight='bold')
ax2.set_title('Fitted Polynomials: Low Degree = Underfitting, High Degree = Overfitting', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10, loc='upper right')
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xlim([0, 2*np.pi])

plt.tight_layout()
plt.show()

## Section 3: Cross-Validation for Model Selection

Rather than a single train/test split, use k-fold cross-validation to get a more robust estimate of test error. This is especially important with small datasets.

In [ ]:
# Use 5-fold cross-validation to select the best polynomial degree
cv_scores_all = []

for degree in degrees:
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('ridge', Ridge(alpha=0.001))
    ])
    
    # Negative MSE is the default scoring (sklearn convention: higher is better)
    cv_scores = -cross_val_score(
        model, X_train, y_train, cv=5, scoring='neg_mean_squared_error'
    )
    cv_rmse = np.sqrt(cv_scores)
    cv_scores_all.append(cv_rmse)
    
    print(f"Degree {degree}: CV RMSE = {cv_rmse.mean():.4f} ± {cv_rmse.std():.4f}")

# Find the best degree
best_degree = degrees[np.argmin([s.mean() for s in cv_scores_all])]
print(f"\nBest degree (by CV): {best_degree}")
print(f"This model has the lowest CV error, so we expect it to generalize best.")

In [ ]:
# Plot CV scores
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

cv_means = [cv_rmse.mean() for cv_rmse in cv_scores_all]
cv_stds = [cv_rmse.std() for cv_rmse in cv_scores_all]

ax.errorbar(degrees, cv_means, yerr=cv_stds, marker='o', linewidth=2.5, markersize=8,
            color='#2A9D8F', capsize=5, capthick=2, label='5-Fold CV RMSE ± std')

# Highlight the best degree
best_idx = np.argmin(cv_means)
ax.scatter([best_degree], [cv_means[best_idx]], s=200, marker='*', color='#F77F00',
           edgecolor='black', linewidth=1.5, zorder=5, label=f'Best: Degree {best_degree}')

ax.set_xlabel('Polynomial Degree', fontsize=12, fontweight='bold')
ax.set_ylabel('Cross-Validation RMSE', fontsize=12, fontweight='bold')
ax.set_title('Model Selection via 5-Fold Cross-Validation', fontsize=13, fontweight='bold')
ax.set_xticks(degrees)
ax.legend(fontsize=11, loc='upper right')
ax.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

print(f"\nCV uses multiple train/test splits, so it's more robust than a single holdout.")

## Section 4: Shuffled-Label Sanity Check

A critical sanity check: train on randomly shuffled labels. If your pipeline still achieves high accuracy on shuffled labels, something is broken (data leakage, wrong evaluation, etc.).

In [ ]:
# Create a regression version to do a cleaner sanity check
# Use the noisy training data and shuffle the target

print("SANITY CHECK: Shuffled-Label Test")
print("=" * 60)

# Train best model on real labels
model_real = Pipeline([
    ('poly', PolynomialFeatures(degree=best_degree, include_bias=False)),
    ('ridge', Ridge(alpha=0.001))
])
model_real.fit(X_train, y_train)
test_rmse_real = np.sqrt(mean_squared_error(y_test, model_real.predict(X_test)))

print(f"\n1. Model on REAL labels:")
print(f"   Degree {best_degree} polynomial")
print(f"   Test RMSE: {test_rmse_real:.4f}")

# Train on shuffled labels
y_train_shuffled = y_train.copy()
np.random.shuffle(y_train_shuffled)

model_shuffled = Pipeline([
    ('poly', PolynomialFeatures(degree=best_degree, include_bias=False)),
    ('ridge', Ridge(alpha=0.001))
])
model_shuffled.fit(X_train, y_train_shuffled)
test_rmse_shuffled = np.sqrt(mean_squared_error(y_test, model_shuffled.predict(X_test)))

print(f"\n2. Model on SHUFFLED labels (same data, random targets):")
print(f"   Test RMSE: {test_rmse_shuffled:.4f}")

print(f"\n3. Sanity Check Result:")
if test_rmse_shuffled > test_rmse_real * 0.9:  # Allow some statistical noise
    print(f"   ⚠️  WARNING: Model performs almost as well on shuffled labels!")
    print(f"   This suggests data leakage or a broken pipeline.")
else:
    print(f"   ✓ PASS: Model fails on shuffled labels (RMSE ratio: {test_rmse_shuffled/test_rmse_real:.2f})")
    print(f"   This is the expected behavior—random targets should yield poor performance.")

In [ ]:
# Visualize the sanity check
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Real vs shuffled RMSE
results = ['Real Labels', 'Shuffled Labels']
rmses = [test_rmse_real, test_rmse_shuffled]
colors_bar = ['#2A9D8F', '#E63946']

bars = ax1.bar(results, rmses, color=colors_bar, alpha=0.7, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('Test RMSE', fontsize=12, fontweight='bold')
ax1.set_title('Sanity Check: Real vs Shuffled Labels', fontsize=13, fontweight='bold')
ax1.set_ylim([0, max(rmses) * 1.2])

# Add value labels
for bar, rmse in zip(bars, rmses):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{rmse:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax1.grid(axis='y', alpha=0.3, linestyle='--')

# Right: Predictions on real vs shuffled
y_pred_real = model_real.predict(X_test)
y_pred_shuffled = model_shuffled.predict(X_test)

ax2.scatter(X_train, y_train, color='black', s=40, alpha=0.5, label='Training data', zorder=3)
ax2.plot(X_test, y_test, 'k--', linewidth=2.5, label='True sin(x)', alpha=0.7, zorder=2)
ax2.plot(X_test, y_pred_real, linewidth=2, color='#2A9D8F', label='Fit to real labels', zorder=2)
ax2.plot(X_test, y_pred_shuffled, linewidth=2, color='#E63946', label='Fit to shuffled labels', linestyle=':', zorder=2)

ax2.set_xlabel('x', fontsize=12, fontweight='bold')
ax2.set_ylabel('y', fontsize=12, fontweight='bold')
ax2.set_title('Model Predictions: Real vs Shuffled Labels', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11, loc='upper right')
ax2.grid(alpha=0.3, linestyle='--')
ax2.set_xlim([0, 2*np.pi])

plt.tight_layout()
plt.show()

## Section 5: Learning Curves

Learning curves show how training and validation accuracy improve as we add more training data. They reveal whether our bottleneck is model capacity (high bias) or data insufficiency (high variance).

In [ ]:
# Generate a larger synthetic dataset for learning curves
X_large = np.linspace(0, 2*np.pi, 200).reshape(-1, 1)
y_large = np.sin(X_large.ravel()) + np.random.normal(0, 0.15, len(X_large))

# Test set
X_test_lc = np.linspace(0, 2*np.pi, 300).reshape(-1, 1)
y_test_lc = np.sin(X_test_lc.ravel())

# Learning curve with best degree
model_lc = Pipeline([
    ('poly', PolynomialFeatures(degree=best_degree, include_bias=False)),
    ('ridge', Ridge(alpha=0.001))
])

train_sizes, train_scores, val_scores = learning_curve(
    model_lc, X_large, y_large, cv=5,
    train_sizes=np.linspace(5, len(X_large), 8),
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

# Convert from negative MSE to RMSE
train_rmse = np.sqrt(-train_scores)
val_rmse = np.sqrt(-val_scores)

train_rmse_mean = train_rmse.mean(axis=1)
train_rmse_std = train_rmse.std(axis=1)
val_rmse_mean = val_rmse.mean(axis=1)
val_rmse_std = val_rmse.std(axis=1)

print(f"Learning Curves (Degree {best_degree}):")
print("-" * 70)
print(f"{'Training Size':<20} {'Train RMSE':<25} {'Val RMSE':<25}")
print("-" * 70)
for i, size in enumerate(train_sizes):
    print(f"{int(size):<20} {train_rmse_mean[i]:.4f} ± {train_rmse_std[i]:.4f}  {val_rmse_mean[i]:.4f} ± {val_rmse_std[i]:.4f}")
print("-" * 70)

In [ ]:
# Plot learning curves
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Plot training curve
ax.plot(train_sizes, train_rmse_mean, marker='o', linewidth=2.5, markersize=8,
        color='#2E86AB', label='Training Error', alpha=0.8)
ax.fill_between(train_sizes, 
                 train_rmse_mean - train_rmse_std,
                 train_rmse_mean + train_rmse_std,
                 alpha=0.2, color='#2E86AB')

# Plot validation curve
ax.plot(train_sizes, val_rmse_mean, marker='s', linewidth=2.5, markersize=8,
        color='#A23B72', label='Validation Error', alpha=0.8)
ax.fill_between(train_sizes,
                 val_rmse_mean - val_rmse_std,
                 val_rmse_mean + val_rmse_std,
                 alpha=0.2, color='#A23B72')

ax.set_xlabel('Training Set Size', fontsize=12, fontweight='bold')
ax.set_ylabel('RMSE', fontsize=12, fontweight='bold')
ax.set_title(f'Learning Curves: How Error Decreases with More Data (Degree {best_degree})', fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

print(f"\nInterpretation:")
print(f"- Training error increases with more data (less overfitting on large sets).")
print(f"- Validation error decreases as we add more data (more signal to learn from).")
print(f"- If the gap between train and val error is large, we have high variance (need more data).")
print(f"- If both errors are high and don't improve, we have high bias (need more capacity or features).")

## What to Notice

1. **The U-curve is real**: Training error always decreases, but test error eventually increases. The sweet spot is where test error is minimized.

2. **Cross-validation is more reliable than holdout**: With limited data, a single train/test split is noisy. CV averages over multiple splits.

3. **Always do the shuffled-label sanity check**: If your model performs well on shuffled labels, you have a serious problem. This catches data leakage and pipeline bugs.

4. **Learning curves diagnose the problem**:
   - Large gap between train and val error → high variance (need more data or simpler model)
   - Both errors high → high bias (need more capacity or better features)

5. **More data is often the best regularizer**: If your learning curves show that val error still decreases at the end, collecting more data will likely help.

6. **Evaluation discipline is non-negotiable**: Always separate train, validation, and test sets. Always report honest test performance, never training performance.